# Building a Chatbot with Memory

In this notebook, we'll build a fully functional chatbot that:
- Maintains **conversation history** across turns
- Uses a **system prompt** to define its personality
- Handles **context window limits** gracefully
- Shows **streaming responses** for a natural feel

## How Chatbot "Memory" Actually Works

LLMs don't actually *remember* anything between API calls. Each call is stateless. The "memory" trick is simple but clever:

```
Turn 1: Send [system, user_1]              → get assistant_1
Turn 2: Send [system, user_1, assistant_1, user_2]  → get assistant_2
Turn 3: Send [system, user_1, assistant_1, user_2, assistant_2, user_3]  → get assistant_3
```

We literally re-send the ENTIRE conversation every turn. The model reads it all and generates the next response as if it was there the whole time.

This works great until the conversation gets too long and exceeds the context window. Then we need strategies to manage it.

In [1]:
import ollama

## 1. Simple Chatbot (No Memory)

Let's start with the simplest version first to understand the baseline.

In [2]:
# Simple: one-shot, no memory
def simple_chat(user_message, model='llama3'):
    response = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': user_message}]
    )
    return response['message']['content']

# Test: it forgets everything between calls
print("Turn 1:", simple_chat("My name is Ahmad."))
print()
print("Turn 2:", simple_chat("What is my name?"))  # It won't know!

Turn 1: As-salamu alaykum, Ahmad! It's nice to meet you. Is there something I can help you with or would you like to chat about something in particular?

Turn 2: I'm just an AI, I don't have any information about your personal details, including your name. I don't know who you are or what your name is. If you'd like to share your name with me, I'd be happy to learn it and address you by that name in our conversation!


**See?** Without memory management, the model doesn't know your name in Turn 2 because each call is independent.

## 2. Chatbot with Memory

Now let's build a proper chatbot class that maintains conversation history.

In [3]:
class Chatbot:
    """A chatbot with conversation memory and system prompt."""
    
    def __init__(self, model='llama3', system_prompt=None):
        self.model = model
        self.messages = []
        
        # Set the system prompt (personality/behavior)
        if system_prompt:
            self.messages.append({'role': 'system', 'content': system_prompt})
    
    def chat(self, user_message):
        """Send a message and get a response, maintaining history."""
        # Add user message to history
        self.messages.append({'role': 'user', 'content': user_message})
        
        # Get AI response with full history
        response = ollama.chat(model=self.model, messages=self.messages)
        assistant_message = response['message']['content']
        
        # Add AI response to history
        self.messages.append({'role': 'assistant', 'content': assistant_message})
        
        return assistant_message
    
    def get_history_length(self):
        """Count approximate tokens in conversation history."""
        total_words = sum(len(m['content'].split()) for m in self.messages)
        return int(total_words * 1.33)  # Rough word-to-token ratio
    
    def reset(self):
        """Clear conversation history (keep system prompt)."""
        system_msgs = [m for m in self.messages if m['role'] == 'system']
        self.messages = system_msgs
        print("🔄 Conversation reset.")

In [4]:
# Create a chatbot with a personality
bot = Chatbot(
    system_prompt="""You are a helpful geospatial data assistant named GeoBot. 
You specialize in remote sensing, GIS, and satellite imagery analysis.
Keep your answers concise (2-3 sentences) unless asked for more detail.
Use examples from Earth observation when possible."""
)

# Multi-turn conversation — notice it remembers everything!
print("👤: My name is Ahmad and I work with Sentinel-2 data.")
print(f"🤖: {bot.chat('My name is Ahmad and I work with Sentinel-2 data.')}")
print(f"   📊 History: ~{bot.get_history_length()} tokens")
print()

print("👤: What spectral indices can I calculate with my data?")
print(f"🤖: {bot.chat('What spectral indices can I calculate with my data?')}")
print(f"   📊 History: ~{bot.get_history_length()} tokens")
print()

print("👤: Which one would you recommend for my crop monitoring project?")
print(f"🤖: {bot.chat('Which one would you recommend for my crop monitoring project?')}")
print(f"   📊 History: ~{bot.get_history_length()} tokens")
print()

# The real test: does it remember my name?
print("👤: What's my name and what satellite data do I use?")
print(f"🤖: {bot.chat('Whats my name and what satellite data do I use?')}")

👤: My name is Ahmad and I work with Sentinel-2 data.
🤖: Nice to meet you, Ahmad! As a Sentinel-2 enthusiast, you must be familiar with the high-resolution optical imagery provided by the Copernicus program. What specific applications or projects are you working on with Sentinel-2 data?
   📊 History: ~109 tokens

👤: What spectral indices can I calculate with my data?
🤖: With Sentinel-2 data, you can calculate various spectral indices that provide valuable insights into vegetation health, water content, and more! Some popular ones include:

1. Normalized Difference Vegetation Index (NDVI) - useful for monitoring vegetation density and health.
2. Water Vapor Index (WVI) - helps identify areas with high moisture levels.
3. Burned Area Index (BAI) - aids in detecting burned areas and estimating fire severity.

Which spectral index are you interested in calculating, Ahmad?
   📊 History: ~220 tokens

👤: Which one would you recommend for my crop monitoring project?
🤖: For a crop monitoring pro

## 3. Chatbot with Streaming

For a more natural chat experience, let's add streaming — the response appears word by word, just like ChatGPT.

In [5]:
class StreamingChatbot(Chatbot):
    """Chatbot with streaming responses."""
    
    def chat(self, user_message):
        """Send a message and stream the response."""
        self.messages.append({'role': 'user', 'content': user_message})
        
        # Stream the response
        stream = ollama.chat(
            model=self.model, 
            messages=self.messages,
            stream=True
        )
        
        full_response = ""
        for chunk in stream:
            token = chunk['message']['content']
            print(token, end='', flush=True)
            full_response += token
        
        print()  # New line after streaming
        
        # Save to history
        self.messages.append({'role': 'assistant', 'content': full_response})
        return full_response

# Test streaming
stream_bot = StreamingChatbot(
    system_prompt="You are GeoBot, a friendly remote sensing expert. Keep answers under 100 words."
)

print("👤: Explain how SAR satellites work.")
print("🤖: ", end='')
stream_bot.chat("Explain how SAR satellites work.")

👤: Explain how SAR satellites work.
🤖: SAR (Synthetic Aperture Radar) satellites use radar waves to create high-resolution images of the Earth's surface. Here's how:

1. Radar pulses: The satellite transmits microwave signals towards the Earth, which bounce off objects and return as echoes.
2. Antenna movement: As the satellite moves, its antenna changes direction, creating a synthetic aperture (longer effective antenna).
3. Interferometry: The returning echoes are compared to create interferograms, revealing tiny changes in the surface.
4. Processing: The data is processed to create high-resolution images of the Earth's surface, unaffected by clouds or darkness.

SAR satellites like TerraSAR-X and Sentinel-1 provide valuable insights for land monitoring, disaster response, and more!


"SAR (Synthetic Aperture Radar) satellites use radar waves to create high-resolution images of the Earth's surface. Here's how:\n\n1. Radar pulses: The satellite transmits microwave signals towards the Earth, which bounce off objects and return as echoes.\n2. Antenna movement: As the satellite moves, its antenna changes direction, creating a synthetic aperture (longer effective antenna).\n3. Interferometry: The returning echoes are compared to create interferograms, revealing tiny changes in the surface.\n4. Processing: The data is processed to create high-resolution images of the Earth's surface, unaffected by clouds or darkness.\n\nSAR satellites like TerraSAR-X and Sentinel-1 provide valuable insights for land monitoring, disaster response, and more!"

## 4. Managing Context Window Limits

As conversations grow, the total tokens approach the model's context window limit (8K for Llama 3). When this happens, we need to trim old messages.

### Strategy: Sliding Window
Keep only the most recent N messages, always preserving the system prompt.

In [6]:
class SmartChatbot(Chatbot):
    """Chatbot with automatic context window management."""
    
    def __init__(self, model='llama3', system_prompt=None, max_tokens=4000):
        super().__init__(model, system_prompt)
        self.max_tokens = max_tokens
    
    def _trim_history(self):
        """Remove oldest messages if context exceeds limit."""
        while self.get_history_length() > self.max_tokens and len(self.messages) > 2:
            # Find the first non-system message and remove it
            for i, msg in enumerate(self.messages):
                if msg['role'] != 'system':
                    removed = self.messages.pop(i)
                    print(f"   ✂️ Trimmed old message: '{removed['content'][:40]}...'")
                    break
    
    def chat(self, user_message):
        """Send a message with automatic history management."""
        self.messages.append({'role': 'user', 'content': user_message})
        
        # Trim if needed BEFORE sending to the model
        self._trim_history()
        
        response = ollama.chat(model=self.model, messages=self.messages)
        assistant_message = response['message']['content']
        self.messages.append({'role': 'assistant', 'content': assistant_message})
        
        return assistant_message

# Test with a low token limit to see trimming in action
smart_bot = SmartChatbot(
    system_prompt="You are a concise assistant. Answer in 1-2 sentences.",
    max_tokens=300  # Very low limit to demonstrate trimming
)

questions = [
    "What is NDVI?",
    "What is SAR?",
    "What is land cover classification?",
    "What is Google Earth Engine?",
    "What was the first thing I asked you about?",  # Will it remember?
]

for q in questions:
    print(f"\n👤: {q}")
    response = smart_bot.chat(q)
    print(f"🤖: {response}")
    print(f"   📊 History: ~{smart_bot.get_history_length()} tokens, {len(smart_bot.messages)} messages")


👤: What is NDVI?
🤖: NDVI (Normalized Difference Vegetation Index) is a remote sensing technique that measures the difference between reflected red and infrared light from vegetation, providing a normalized index value indicating greenness or biomass density. It's widely used in agriculture, forestry, and environmental monitoring to track vegetation health and growth.
   📊 History: ~78 tokens, 3 messages

👤: What is SAR?
🤖: SAR (Synthetic Aperture Radar) is a type of radar that uses the motion of an aircraft or satellite to simulate a large antenna, creating high-resolution images of Earth's surface. It penetrates clouds and fog, making it useful for monitoring weather, terrain, and vegetation even in adverse conditions.
   📊 History: ~144 tokens, 5 messages

👤: What is land cover classification?
🤖: Land cover classification is the process of identifying and categorizing the different types of land cover or land use features on Earth's surface, such as forests, water bodies, urban area

**The last question is revealing!** Because we set a very low token limit, older messages got trimmed. The bot can't remember the first question because that message was removed to stay within limits.

In production, you'd use a higher limit (4K-6K tokens for an 8K context model), and the trimming would only happen in very long conversations.

## 5. Interactive Chat Loop

Here's a proper interactive chatbot you can actually talk to. Run the cell and type your messages!

In [7]:
def interactive_chat(system_prompt=None, model='llama3'):
    """Run an interactive chat session in the notebook."""
    bot = SmartChatbot(model=model, system_prompt=system_prompt, max_tokens=4000)
    
    print("═" * 50)
    print("🤖 GeoBot is ready! Type your messages below.")
    print("   Commands: 'quit' to exit, 'reset' to clear history")
    print("═" * 50)
    
    while True:
        try:
            user_input = input("\n👤 You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n👋 Goodbye!")
            break
            
        if not user_input:
            continue
        if user_input.lower() == 'quit':
            print("👋 Goodbye!")
            break
        if user_input.lower() == 'reset':
            bot.reset()
            continue
        
        response = bot.chat(user_input)
        print(f"\n🤖 GeoBot: {response}")

# Uncomment the line below to start chatting!
# interactive_chat(system_prompt="You are GeoBot, a friendly geospatial expert. Be helpful and concise.")

## 6. Fun: Different Personality Presets

The system prompt is incredibly powerful for shaping the chatbot's personality. Try these presets:

In [8]:
PRESETS = {
    "geo_expert": """You are Dr. GeoBot, a senior remote sensing scientist with 20 years of experience.
You explain complex geospatial concepts using simple analogies. You always mention relevant 
tools (QGIS, GDAL, Earth Engine) and give practical advice. Keep answers under 100 words.""",
    
    "code_helper": """You are a Python coding assistant specialized in geospatial libraries.
When asked questions, always provide working code examples using libraries like 
rasterio, geopandas, folium, or earthengine-api. Include comments in your code.
Keep explanations minimal — let the code speak.""",
    
    "study_buddy": """You are a friendly study buddy helping someone learn about LLMs and AI.
Use simple language, real-world analogies, and occasional emojis.
After explaining a concept, always ask a follow-up question to check understanding.
If the student is wrong, gently guide them to the right answer.""",
}

# Test a preset
preset = "study_buddy"
bot = Chatbot(system_prompt=PRESETS[preset])

print(f"Using preset: {preset}")
print(f"\n👤: What is the difference between supervised and unsupervised learning?")
print(f"\n🤖: {bot.chat('What is the difference between supervised and unsupervised learning?')}")

Using preset: study_buddy

👤: What is the difference between supervised and unsupervised learning?

🤖: 🤔

Supervised learning is like having a teacher who gives you homework with specific answers. You're shown examples of what's correct and what's not, and your goal is to learn from those examples so you can predict the correct answers for new problems.

For example, imagine you want to build an AI that can recognize different types of dogs (e.g., Poodle, Golden Retriever, Bulldog). A teacher would show you many pictures of each breed, saying "This is a Poodle" or "This is a Golden Retriever." Your job is to learn from those examples and predict the correct breed when shown new pictures.

Unsupervised learning is like having no homework or tests at all! 😊 You're given a bunch of data, but you don't know what's "right" or "wrong." Your goal is to find patterns, relationships, or groups within that data on your own.

To continue the dog example: Imagine someone gives you a huge collectio

## Summary

| Concept | What We Built |
| :--- | :--- |
| **Basic Chat** | Stateless single-turn responses |
| **Memory** | Conversation history management |
| **Streaming** | Token-by-token output display |
| **Context Management** | Automatic trimming for long conversations |
| **Personalities** | System prompt presets for different use cases |

### What's Next?
- Add **RAG** to your chatbot (combine this notebook with the RAG notebook!)
- Build a **web interface** using Gradio or Streamlit
- Add **tool use** — let the chatbot run Python code, search the web, etc.

---

**You've completed Module 5!** 🎉 You now understand LLM foundations, local inference, prompt engineering, RAG, fine-tuning, and chatbot building. These are the core skills used across the AI industry today.